# Análisis Exploratorio

Lectura compras.csv

In [0]:
# Read the CSV file, skipping the header
with open('/Volumes/workspace/default/prueba_bavaria/compras.csv', 'r') as f:
    header = f.readline()
    rows = f.readlines()

# Replace | with , in the rows
rows = [row.replace('|', ',') for row in rows]

# Write the header and modified rows to a new CSV file
output_path = '/Volumes/workspace/default/prueba_bavaria/compras_clean.csv'
with open(output_path, 'w') as f:
    f.write(header)
    f.writelines(rows)

# Read the cleaned CSV into a Spark DataFrame
df_compras = spark.read.format("csv").option("header", "true").load(output_path)

# Create Databricks SQL table
df_compras.write.format("delta").mode("overwrite").saveAsTable("workspace.default.prueba_bavaria_compras")

In [0]:
display(df_compras)
df_compras.printSchema()

Lectura productos.csv

In [0]:
# Read the CSV file, skipping the header
with open('/Volumes/workspace/default/prueba_bavaria/productos.csv', 'r') as f:
    header = f.readline()
    rows = f.readlines()

# Replace | with , in the rows
rows = [row.replace('|', ',') for row in rows]

# Write the header and modified rows to a new CSV file
output_path = '/Volumes/workspace/default/prueba_bavaria/productos_clean.csv'
with open(output_path, 'w') as f:
    f.write(header)
    f.writelines(rows)

# Read the cleaned CSV into a Spark DataFrame
df_productos = spark.read.format("csv").option("header", "true").load(output_path)

# Create Databricks SQL table
df_productos.write.format("delta").mode("overwrite").saveAsTable("workspace.default.prueba_bavaria_productos")

In [0]:
display(df_productos)
df_productos.printSchema()

Join Tablas

In [0]:
df_compras = spark.table("workspace.default.prueba_bavaria_compras")
df_productos = spark.table("workspace.default.prueba_bavaria_productos")

df = df_compras.join(df_productos, on="SKU_ID", how="left")

In [0]:
from pyspark.sql.functions import col, to_date

df = df.withColumn("GMV", col("GMV").cast("double")) \
       .withColumn("Unit_Quantity", col("Unit_Quantity").cast("int")) \
       .withColumn("Volume", col("Volume").cast("double")) \
       .withColumn("SKU SIZE", col("SKU SIZE").cast("double")) \
       .withColumn("Order_Date", to_date("Order_Date"))

In [0]:
df.printSchema()
df.describe().show()

In [0]:
df.filter(col("Category").isNull()).count()

Métricas de Negocio

In [0]:
from pyspark.sql.functions import countDistinct, sum

df.select(
    countDistinct("Customer").alias("clientes"),
    countDistinct("POC").alias("puntos_venta"),
    sum("GMV").alias("ventas_totales"),
    sum("Unit_Quantity").alias("unidades_totales")
).show()

#Perfiles de Cliente

Features por Cliente

In [0]:
from pyspark.sql.functions import avg, countDistinct

customer_df = df.groupBy("Customer").agg(
    countDistinct("Order_Date").alias("frecuencia"),
    sum("GMV").alias("gasto_total"),
    avg("GMV").alias("ticket_promedio"),
    sum("Unit_Quantity").alias("volumen_total"),
    countDistinct("Brand").alias("diversidad_marcas"),
    countDistinct("POC").alias("diversidad_puntos")
)

Preparación K-Means

In [0]:
from pyspark.ml.feature import VectorAssembler

features = [
    "frecuencia",
    "gasto_total",
    "ticket_promedio",
    "volumen_total",
    "diversidad_marcas",
    "diversidad_puntos"
]

assembler = VectorAssembler(inputCols=features, outputCol="features")

customer_vector = assembler.transform(customer_df).select("Customer", "features")

Escalado para Clustering

In [0]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

scaler_model = scaler.fit(customer_vector)
customer_scaled = scaler_model.transform(customer_vector)

Aplicación K-Means

In [0]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(k=4, seed=42, featuresCol="scaled_features")
model = kmeans.fit(customer_scaled)

clusters = model.transform(customer_scaled)

In [0]:
final_df = clusters.join(customer_df, on="Customer")

final_df.groupBy("prediction").agg(
    avg("frecuencia"),
    avg("gasto_total"),
    avg("ticket_promedio"),
    avg("volumen_total"),
    avg("diversidad_marcas")
).show()

###Interepretación de Perfiles

**Cluster 0: Clientes de alto valor**

Alto gasto
Alta frecuencia
Alto ticket

Insight:
Este grupo representa clientes de alto valor, ideales para estrategias de fidelización.

**Cluster 1: Clientes Frecuentes**

Compran seguido
Gastan poco por compra

Insight:
Sensibles a promociones y descuentos.

**Cluster 2: Clientes Ocasionales**

Baja frecuencia
Bajo gasto

Insight:
Oportunidad de activación mediante campañas.

**Cluster 3: Exploradores de Marca**

Alta diversidad de marcas

Insight:
Baja lealtad, alto potencial de switching.


In [0]:
df.groupBy("Order_Date").sum("GMV")

In [0]:
%sql
SELECT * FROM workspace.default.prueba_bavaria_compras

Databricks visualization. Run in Databricks to view.

In [0]:
df_pandas = df.toPandas()
display(df_pandas)
   